In [0]:
from pyspark.sql import functions as f
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_etl.validations_ETL as validations
from lib_etl.s3 import etl_input_data_validator
from lib.s3 import etl_input_table_validator
from lib.job_manager import load_config, split_config

In [0]:
%run ../../config/utils

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)
run_as_date = dbutils.widgets.get("run_as_date")

### Transform 

In [0]:
recency_lookback_duration = data_paths.get("recency_lookback_duration", {})
etl_input_table_validator(
    silver_master_item,  silver_master_brand,
    recency_lookback_duration=recency_lookback_duration,
    spark=spark
)

In [0]:
ah5 = spark.table(fs_ah5_cd_category_dna_full)

In [0]:
item = spark.table(silver_master_item)
brands = spark.table(silver_master_brand)
ah5 = spark.read.parquet(data_paths["intermediate"]["AH5"])

window = Window.orderBy(col("BRAND_DESC").asc())

distinct_brands = (
    brands.drop("CASE_EXPRESSION", "ARTICLE_NBR")
    .withColumnRenamed("BRAND", "BRAND_CD")
    .withColumn("BRAND_DESC", col("BRAND_CD"))
    .distinct()
)

brands_join = distinct_brands.drop("BRAND_CD").withColumn(
    "rn", row_number().over(window)
)

# could make dynamic to make sure it's always greater than, but the count just needs to always be greater than number of distinct brands
ah5_join = ah5
for i in range(1, 5):
    ah5_join = ah5_join.union(ah5_join)

assert (
    ah5_join.count() > brands_join.select("brand_desc").distinct().count()
)

ah5_window = Window.orderBy(col("AH5_CD").asc())

ah5_join = ah5_join.withColumn("rn", row_number().over(ah5_window))

final = (
    brands_join.distinct()
    .join(ah5_join.drop("ah5_cd", "ah5_desc"), "rn")
    .select(
        ["rn", "BRAND_DESC"]
        + ah5_join.drop("ah5_cd", "ah5_desc", "rn").columns
    )
    .withColumnRenamed("rn", "BRAND_CD")
)

item_with_brand = (
    brands.drop("CASE_EXPRESSION")
    .join(brands_join, col("BRAND") == col("BRAND_DESC"))
    .withColumnRenamed("rn", "BRAND_CD")
    .drop("brand")
)

df_item_brand = item.join(item_with_brand, "ARTICLE_NBR", "left")
df_item_brand.createOrReplaceTempView("source")

In [0]:
validations.validate_table(
        spark, "intermediate", 'item_with_brand', config_validation, df_item_brand, stats_etl_path
    )

### Merge

In [0]:
df_item_brand.write.mode("overwrite").saveAsTable(silver_ad_hoc_master_item_with_brand)

if archive_flag:
    save_archive(df_item_brand, silver_ad_hoc_master_item_with_brand_archive, run_as_date)